# Notebook 01 — Data Preprocessing & Embedding Pipeline

This notebook demonstrates the full data preparation pipeline for the AI-Based Technical Standards Compliance Checker.

**Steps covered:**
1. Load and parse the sample engineering document
2. Extract and clean paragraphs
3. Load compliance clauses from CSV
4. Generate sentence embeddings (all-MiniLM-L6-v2)
5. Build FAISS vector index
6. t-SNE visualisation of embedding space
7. Similarity heatmap analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cos
from sklearn.manifold import TSNE

from src.embedder import Embedder
from src.search import DocumentIndex

plt.style.use('seaborn-v0_8-whitegrid')
print('All imports successful')

## 1. Load Sample Document

In [ ]:
doc_path = Path('../data/test_document.txt')
raw_text = doc_path.read_text(encoding='utf-8')
print(f'Document: {doc_path.name}')
print(f'Characters: {len(raw_text):,}')
print(f'Lines:      {len(raw_text.splitlines()):,}')
print('\n--- Preview (first 400 chars) ---')
print(raw_text[:400])

In [ ]:
# Extract paragraphs
paragraphs = [p.strip() for p in raw_text.split('\n\n') if len(p.strip()) > 40]
print(f'Paragraphs extracted: {len(paragraphs)}')
print('\nParagraph length distribution (chars):')
lengths = [len(p) for p in paragraphs]
pd.Series(lengths).describe().to_frame('length').T

## 2. Load Compliance Clauses

In [ ]:
clauses_df = pd.read_csv('../data/sample_standard.csv')
print(f'Total clauses loaded: {len(clauses_df)}')
display(clauses_df.groupby(['standard_id','clause_type']).size().rename('count').reset_index())

In [ ]:
# Visualise clause distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clauses_df.groupby('standard_id').size().plot(kind='bar', ax=axes[0],
    color=['#1155AA','#4A148C','#006B5A','#B45309','#7F1D1D','#006064'],
    edgecolor='white', linewidth=0.8)
axes[0].set_title('Clauses per Standard', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Standard ID')
axes[0].set_ylabel('Number of Clauses')
axes[0].tick_params(axis='x', rotation=0)

clauses_df['clause_type'].value_counts().plot(kind='pie', ax=axes[1],
    colors=['#1155AA','#1B8A4F','#E67E22','#C0392B'],
    autopct='%1.0f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Clause Types', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.suptitle('Compliance Clause Registry — Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('clause_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Generate Embeddings — BERT vs TF-IDF Comparison

In [ ]:
emb = Embedder()
print('Sentence-Transformer model loaded: all-MiniLM-L6-v2')
print(f'Embedding dimension: {emb.embedding_dim}')

In [ ]:
# Classic vocabulary-mismatch demo
pairs = [
    ('differential output voltage at MDI shall be between 0.67V and 1.33V',
     'PHY signal level at RJ-45 connector measures 1.1V'),
    ('traces carrying more than 1A shall have minimum width of 0.3mm',
     'gate driver conductors are 1.5mm wide rated for 15A'),
    ('test records shall be maintained for minimum 10 years',
     'PLM system retains commissioning certificates for 7 years'),
    ('auto-negotiation shall be implemented and enabled by default',
     'The interface supports 10BASE-T and 100BASE-TX auto-negotiation'),
]

results = []
for clause, doc in pairs:
    bert_sim = emb.cosine_similarity(emb.encode(clause), emb.encode(doc))
    tfidf_m  = TfidfVectorizer().fit_transform([clause, doc])
    tf_sim   = float(sk_cos(tfidf_m[0], tfidf_m[1])[0][0])
    results.append({'Clause (truncated)': clause[:55]+'...', 'BERT Sim': round(bert_sim,3), 'TF-IDF Sim': round(tf_sim,3), 'BERT gain': round(bert_sim-tf_sim,3)})

results_df = pd.DataFrame(results)
display(results_df)
print(f'\nAverage BERT similarity:  {results_df["BERT Sim"].mean():.3f}')
print(f'Average TF-IDF similarity: {results_df["TF-IDF Sim"].mean():.3f}')
print(f'Average improvement:       +{results_df["BERT gain"].mean():.3f}')

In [ ]:
# Bar chart: BERT vs TF-IDF
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results_df))
w = 0.35
bars1 = ax.bar(x - w/2, results_df['BERT Sim'],   w, label='BERT (Semantic)', color='#1155AA', alpha=0.9)
bars2 = ax.bar(x + w/2, results_df['TF-IDF Sim'], w, label='TF-IDF (Keyword)', color='#C0392B', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([f'Pair {i+1}' for i in range(len(x))], fontsize=11)
ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('BERT Semantic Embeddings vs TF-IDF\n(same meaning, different vocabulary)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(0.65, color='orange', linestyle='--', alpha=0.7, label='Match threshold')
for bar in bars1:
    ax.annotate(f'{bar.get_height():.2f}', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02),
                ha='center', va='bottom', fontsize=10, color='#1155AA', fontweight='bold')
for bar in bars2:
    ax.annotate(f'{bar.get_height():.2f}', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02),
                ha='center', va='bottom', fontsize=10, color='#C0392B', fontweight='bold')
plt.tight_layout()
plt.savefig('bert_vs_tfidf.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Build FAISS Index & Retrieve Evidence

In [ ]:
index = DocumentIndex(emb)
index.build(paragraphs)
print(f'FAISS index: {index.paragraph_count} vectors indexed')

ieee_clauses = clauses_df[clauses_df['standard_id']=='IEEE']
print('\nRetrieval results for all IEEE 802.3 clauses:')
for _, row in ieee_clauses.iterrows():
    match = index.best_match(row['requirement_text'])
    score = match.score if match else 0.0
    status = 'PASS' if score >= 0.80 else 'WARN' if score >= 0.65 else 'FAIL'
    print(f'  §{row["clause_id"]:12s}  sim={score:.3f}  [{status}]  {row["requirement_text"][:60]}...')

## 5. t-SNE Embedding Space Visualisation

In [ ]:
all_texts  = clauses_df['requirement_text'].tolist() + paragraphs[:25]
all_labels = ['clause']*len(clauses_df) + ['document']*25
all_stds   = clauses_df['standard_id'].tolist() + ['doc']*25

all_vecs = emb.encode(all_texts)
tsne     = TSNE(n_components=2, random_state=42, perplexity=min(20, len(all_vecs)-1))
coords   = tsne.fit_transform(all_vecs)

std_colors = {'IEEE':'#1155AA','IPC':'#4A148C','ISO':'#006B5A','NEC':'#B45309','BIS':'#7F1D1D','ASME':'#006064','doc':'#AAAAAA'}

fig, ax = plt.subplots(figsize=(13, 8))
for std in set(all_stds):
    mask = np.array(all_stds) == std
    marker = 'X' if std != 'doc' else 'o'
    size   = 120 if std != 'doc' else 50
    alpha  = 0.9 if std != 'doc' else 0.5
    ax.scatter(coords[mask,0], coords[mask,1], c=std_colors.get(std,'gray'),
               label=std, marker=marker, s=size, alpha=alpha, edgecolors='white', linewidth=0.5)

ax.set_title('t-SNE Visualisation — Compliance Clauses & Design Document Paragraphs\n'
             'Stars=clauses, Circles=document paragraphs; proximity=semantic similarity',
             fontsize=12, fontweight='bold')
ax.legend(title='Standard / Source', fontsize=10, title_fontsize=10)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
plt.tight_layout()
plt.savefig('tsne_all_standards.png', dpi=150, bbox_inches='tight')
plt.show()